# Human Embryo Time-Lapse Classification — CNN + LSTM

**Dataset**: Zenodo record 6390798 — *A time-lapse embryo dataset for morphokinetic parameter prediction*

## Key Design Differences from Frame-Level CNN Baseline

| Property | Frame-level CNN (baseline) | **This notebook: CNN + LSTM** |
|---|---|---|
| Input unit | Single frame | Sequence of consecutive frames |
| Temporal context | None | Modelled by bidirectional LSTM |
| Sequence length | 1 | **8 frames** (small to fit GPU) |
| Embryo sub-sampling | All embryos | Subset of embryos (frames within sequences are **never** skipped) |
| Backbone | CNN only | CNN feature extractor → LSTM → head |

### Experiment Plan

| Phase | Backbones | Loss |
|---|---|---|
| A | MobileNetV2, ResNet18, ResNet50 | Sparse Categorical Cross-Entropy (SCCE) |
| B | Same 3 backbones | L_total = SCCE + λ · L_ordinal (custom ordinal loss) |

In [1]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os, warnings, random
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.models as tv_models
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
torch.backends.cudnn.benchmark = True

Device: cuda


In [2]:
# ── Cell 2: Global Config ─────────────────────────────────────────────────────
DATA_DIR    = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/'
ANN_DIR     = '/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations/embryo_dataset_annotations'

# ── Sequence-level config ─────────────────────────────────────────────────────
# SEQ_LEN = number of consecutive frames per sequence (kept small to fit VRAM).
# We do NOT skip frames WITHIN a sequence; to reduce total data we cap
# the number of embryos used (MAX_EMBRYOS).  All frames of a chosen
# embryo's phase interval are available for sequence extraction.
SEQ_LEN     = 8      # frames per sequence — small to fit GPU
SEQ_STRIDE  = 4      # sliding-window stride (overlap = SEQ_LEN - SEQ_STRIDE)
MAX_EMBRYOS = 60     # cap embryos to limit dataset size; set None for all

SAMPLE_RATE = 3      # frame sampling inside each phase (1 = every frame)
                     # lower → more frames per phase interval

# ── General training config ───────────────────────────────────────────────────
BATCH_SIZE   = 16    # sequences per batch (each = SEQ_LEN frames)
NUM_EPOCHS   = 5
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LAMBDA       = 0.5   # ordinal-loss weight in L_total
SEED         = 42
IMG_EXTS     = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

# ── LSTM hyper-params ─────────────────────────────────────────────────────────
LSTM_HIDDEN  = 256
LSTM_LAYERS  = 2
BIDIRECTIONAL = True
DROPOUT       = 0.3

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print('Config loaded.')
print(f'  SEQ_LEN={SEQ_LEN}, SEQ_STRIDE={SEQ_STRIDE}, MAX_EMBRYOS={MAX_EMBRYOS}')
print(f'  BATCH_SIZE={BATCH_SIZE}, LSTM_HIDDEN={LSTM_HIDDEN}, BIDIRECTIONAL={BIDIRECTIONAL}')

Config loaded.
  SEQ_LEN=8, SEQ_STRIDE=4, MAX_EMBRYOS=60
  BATCH_SIZE=16, LSTM_HIDDEN=256, BIDIRECTIONAL=True


In [3]:
# ── Cell 3: Label Definitions ─────────────────────────────────────────────────
PHASES = [
    'tPB2','tPNa','tPNf','t2','t3','t4','t5','t6',
    't7','t8','t9+','tM','tSB','tB','tEB','tHB'
]
label_map = {p: i for i, p in enumerate(PHASES)}
print('Phases:', PHASES)
print('Num classes (raw):', len(PHASES))

Phases: ['tPB2', 'tPNa', 'tPNf', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9+', 'tM', 'tSB', 'tB', 'tEB', 'tHB']
Num classes (raw): 16


In [4]:
# ── Cell 4: Build Frame-Level Dataframe ───────────────────────────────────────
# Same logic as the baseline notebook — we list every sampled frame path
# along with its label and embryo ID.  Sequence grouping happens in Cell 7.

def build_dataframe(sample_rate=SAMPLE_RATE, max_embryos=MAX_EMBRYOS):
    """
    Returns a DataFrame with columns [image, label, embryo, frame_idx].

    frame_idx is the *original* frame index within the embryo folder so that
    sequences can be reconstructed in chronological order.

    max_embryos: if not None, randomly sample this many embryos from the full
                 set to reduce total data volume.  No frames are skipped WITHIN
                 any chosen embryo.
    """
    all_csv = sorted([f for f in os.listdir(ANN_DIR) if f.endswith('.csv')])

    # Optional embryo sub-sampling (keeps frames intact within chosen embryos)
    if max_embryos is not None and max_embryos < len(all_csv):
        rng = np.random.default_rng(SEED)
        all_csv = list(rng.choice(all_csv, size=max_embryos, replace=False))

    data = []
    for csv_file in tqdm(all_csv, desc='Scanning annotations'):
        embryo_id  = csv_file.replace('_phases.csv', '')
        ann_path   = os.path.join(ANN_DIR, csv_file)
        img_folder = os.path.join(DATA_DIR, embryo_id)

        if not os.path.exists(img_folder):
            continue

        # Only real image files; skip sub-directories (e.g. F0/)
        images = sorted([
            f for f in os.listdir(img_folder)
            if os.path.isfile(os.path.join(img_folder, f))
            and f.lower().endswith(IMG_EXTS)
        ])
        if not images:
            continue

        ann_df = pd.read_csv(ann_path, header=None)
        ann_df.columns = ['phase', 'start', 'end']

        for _, row in ann_df.iterrows():
            if row['phase'] not in label_map:
                continue
            label = label_map[row['phase']]
            # Sampled frames (skip sample_rate-1 frames between collected ones)
            for frame in range(int(row['start']), int(row['end']), sample_rate):
                if frame < len(images):
                    img_path = os.path.join(img_folder, images[frame])
                    data.append({
                        'image'     : img_path,
                        'label'     : label,
                        'embryo'    : embryo_id,
                        'frame_idx' : frame,
                    })

    df = pd.DataFrame(data)
    print(f'Frame-level rows: {len(df)}  |  Embryos: {df["embryo"].nunique()}')
    return df


df = build_dataframe()
print(df.head())

Scanning annotations: 100%|██████████| 60/60 [00:11<00:00,  5.37it/s]

Frame-level rows: 8882  |  Embryos: 60
                                               image  label    embryo  \
0  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  OJ319-10   
1  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  OJ319-10   
2  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  OJ319-10   
3  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  OJ319-10   
4  /kaggle/input/datasets/abhishekbuddiga06/embry...      0  OJ319-10   

   frame_idx  
0         12  
1         15  
2         18  
3         21  
4         24  


In [5]:
# ── Cell 5: Embryo-level Train / Val / Test Split ─────────────────────────────
# Split is done at the embryo level to prevent data leakage.
# (Same protocol as the baseline notebook.)

embryos = df['embryo'].unique()
train_ids, temp_ids = train_test_split(embryos, test_size=0.30, random_state=SEED)
val_ids,  test_ids  = train_test_split(temp_ids, test_size=0.50, random_state=SEED)

train_df = df[df.embryo.isin(train_ids)].reset_index(drop=True)
val_df   = df[df.embryo.isin(val_ids)].reset_index(drop=True)
test_df  = df[df.embryo.isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)} frames  Val: {len(val_df)} frames  Test: {len(test_df)} frames')

Train: 6082 frames  Val: 1333 frames  Test: 1467 frames


In [6]:
# ── Cell 6: Merge Rare Class (tHB → tEB) ─────────────────────────────────────
# Class 15 (tHB) has < 15 samples; merge into class 14 (tEB) — same fix as
# the baseline notebook.

for split in [train_df, val_df, test_df]:
    split.loc[split['label'] == 15, 'label'] = 14

NUM_CLASSES = 15
print('Num classes after merge:', NUM_CLASSES)
print(train_df['label'].value_counts().sort_index())

Num classes after merge: 15
label
0     184
1     920
2     140
3     614
4     120
5     584
6     105
7     192
8     166
9     723
10    997
11    366
12    360
13    217
14    394
Name: count, dtype: int64


In [7]:
# ── Cell 7: Build Sequences from Frame-Level DataFrames ───────────────────────
#
# Strategy
# --------
# For each (embryo, label) group we have a list of frames in chronological
# order.  We slide a window of SEQ_LEN over those frames with stride
# SEQ_STRIDE.  Frames within each window are NEVER skipped — temporal
# continuity is fully preserved.
#
# If a group has fewer than SEQ_LEN frames we pad by repeating the last
# frame (right-pad) so every sequence is exactly SEQ_LEN long.
#
# Returns a list of (frame_path_list, label) tuples.

def build_sequences(frame_df, seq_len=SEQ_LEN, stride=SEQ_STRIDE):
    """
    Parameters
    ----------
    frame_df : pd.DataFrame  — must have columns [image, label, embryo, frame_idx]
    seq_len  : int           — number of consecutive frames per sequence
    stride   : int           — sliding-window stride

    Returns
    -------
    sequences : list of (List[str], int) — (frame_paths, class_label)
    """
    sequences = []
    # Group by embryo + label (i.e. per phase-interval of that embryo)
    grouped = frame_df.sort_values(['embryo', 'label', 'frame_idx']) \
                       .groupby(['embryo', 'label'])

    for (embryo, label), grp in grouped:
        paths = grp['image'].tolist()  # already sorted by frame_idx
        n     = len(paths)

        if n < seq_len:
            # Pad by repeating the last frame
            padded = paths + [paths[-1]] * (seq_len - n)
            sequences.append((padded, int(label)))
        else:
            # Sliding window — no frames skipped inside each window
            for start in range(0, n - seq_len + 1, stride):
                window = paths[start: start + seq_len]
                sequences.append((window, int(label)))

    return sequences


train_seqs = build_sequences(train_df)
val_seqs   = build_sequences(val_df)
test_seqs  = build_sequences(test_df)

print(f'Sequences — Train: {len(train_seqs)}  Val: {len(val_seqs)}  Test: {len(test_seqs)}')
print(f'Each sequence is {SEQ_LEN} consecutive frames  (stride={SEQ_STRIDE})')

# Quick sanity: check a single sequence
sample_paths, sample_label = train_seqs[0]
print(f'\nExample sequence (label={sample_label}):')
for p in sample_paths:
    print(' ', p)

Sequences — Train: 1207  Val: 265  Test: 300
Each sequence is 8 consecutive frames  (stride=4)

Example sequence (label=0):
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN112.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN115.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN118.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN120.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN123.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset/BA25-7/D2013.01.14_S0704_I132_WELL7_RUN123.jpeg
  /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/em

In [8]:
# ── Cell 8: Class Weights (sequence-level) ────────────────────────────────────
train_labels = [lbl for _, lbl in train_seqs]

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print('Class weights computed.')

Class weights computed.


In [9]:
# ── Cell 9: Sequence Dataset & DataLoaders ────────────────────────────────────

class EmbryoSequenceDataset(Dataset):
    """
    Each sample is a temporal sequence of SEQ_LEN frames from one embryo phase.

    Returns
    -------
    frames : Tensor  shape (SEQ_LEN, C, H, W)
    label  : int
    """
    def __init__(self, sequences, transform=None):
        self.sequences = sequences
        self.transform = transform

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        frame_paths, label = self.sequences[idx]
        frames = []
        for path in frame_paths:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)
        frames = torch.stack(frames)  # (SEQ_LEN, C, H, W)
        return frames, label


def make_loaders(img_size=112, batch_size=BATCH_SIZE):
    """
    Build train / val / test DataLoaders.

    img_size is set smaller than the CNN baseline (112 vs 224) to
    compensate for the SEQ_LEN-fold increase in tensor size per batch.
    """
    train_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    common = dict(batch_size=batch_size, num_workers=4, pin_memory=True,
                  drop_last=True)
    return (
        DataLoader(EmbryoSequenceDataset(train_seqs, train_tf), shuffle=True,  **common),
        DataLoader(EmbryoSequenceDataset(val_seqs,   eval_tf),  shuffle=False, **common),
        DataLoader(EmbryoSequenceDataset(test_seqs,  eval_tf),  shuffle=False, **common),
    )

print('EmbryoSequenceDataset and make_loaders() ready.')

EmbryoSequenceDataset and make_loaders() ready.


## Custom Loss Function — Mathematical Derivation

### 1. Baseline: Sparse Categorical Cross-Entropy (SCCE)

Given logits **z** ∈ ℝᴷ and true label *y* ∈ {0,…,K−1}:

$$\mathcal{L}_{\text{SCCE}}(\mathbf{z},y) = -\log\,p_y, \qquad p_k = \frac{e^{z_k}}{\sum_{j}e^{z_j}}$$

### 2. Custom Component: Ordinal Distance Loss

Embryo phases are **ordered** (tPB2 → … → tEB). We penalise predictions by their ordinal distance to the true class:

$$\boxed{\mathcal{L}_{\text{ordinal}}(\mathbf{z},y) = \frac{1}{K-1}\sum_{k=0}^{K-1} p_k\,|k - y|}$$

#### Total loss
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{SCCE}} + \lambda\,\mathcal{L}_{\text{ordinal}}, \qquad \lambda = 0.5$$

### 3. Why the same loss works well for LSTM sequences

The LSTM's last hidden state aggregates the entire sequence's temporal context before the classifier head applies these losses.  The ordinal penalty therefore discourages the model from confusing temporally *distant* phases (e.g., predicting early-cleavage when the embryo is at blastocyst stage), which is a natural fit for time-lapse modelling.

In [10]:
# ── Cell 10: Loss Functions ───────────────────────────────────────────────────

class SCCELoss(nn.Module):
    """Weighted Sparse Categorical Cross-Entropy with label smoothing."""
    def __init__(self, class_weights, smoothing=0.1):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights,
                                      label_smoothing=smoothing)

    def forward(self, logits, targets):
        return self.ce(logits, targets)


class OrdinalDistanceLoss(nn.Module):
    """Expected ordinal distance under the predicted distribution."""
    def __init__(self, num_classes):
        super().__init__()
        self.K = num_classes
        self.register_buffer('idx',
            torch.arange(num_classes, dtype=torch.float32))

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)               # (B, K)
        t_f   = targets.float().unsqueeze(1)            # (B, 1)
        idx   = self.idx.to(logits.device)              # ensure same device
        dist  = (idx.unsqueeze(0) - t_f).abs()          # (B, K)
        loss  = (probs * dist).sum(dim=1) / (self.K - 1)
        return loss.mean()


class TotalLoss(nn.Module):
    """L_total = SCCE + lambda * L_ordinal"""
    def __init__(self, class_weights, num_classes, lam=LAMBDA, smoothing=0.1):
        super().__init__()
        self.scce    = SCCELoss(class_weights, smoothing)
        self.ordinal = OrdinalDistanceLoss(num_classes)
        self.lam     = lam

    def forward(self, logits, targets):
        return self.scce(logits, targets) + self.lam * self.ordinal(logits, targets)


print('Loss classes: SCCELoss  |  OrdinalDistanceLoss  |  TotalLoss')

Loss classes: SCCELoss  |  OrdinalDistanceLoss  |  TotalLoss


In [11]:
# ── Cell 11: CNN Backbone Factory ─────────────────────────────────────────────
# Each backbone is truncated at its global-average-pool output so that it
# produces a 1-D feature vector per frame.  The LSTM receives a sequence of
# these vectors.

def get_backbone(name: str):
    """
    Returns (backbone_module, feature_dim, recommended_img_size).

    backbone_module  : nn.Module that maps (B, C, H, W) → (B, feature_dim)
    feature_dim      : int — size of the per-frame feature vector
    recommended_img_size : int — spatial resolution to feed the backbone
    """
    name = name.lower()

    # ── MobileNetV2 ───────────────────────────────────────────────────────────
    if name == 'mobilenetv2':
        m = tv_models.mobilenet_v2(
            weights=tv_models.MobileNet_V2_Weights.DEFAULT)
        # Remove the classifier; keep everything up to AdaptiveAvgPool
        backbone = nn.Sequential(
            m.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1),
        )
        feat_dim = 1280
        return backbone, feat_dim, 112

    # ── ResNet18 ──────────────────────────────────────────────────────────────
    if name == 'resnet18':
        m = tv_models.resnet18(
            weights=tv_models.ResNet18_Weights.DEFAULT)
        backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            m.avgpool,
            nn.Flatten(1),
        )
        feat_dim = 512
        return backbone, feat_dim, 112

    # ── ResNet50 ──────────────────────────────────────────────────────────────
    if name == 'resnet50':
        m = tv_models.resnet50(
            weights=tv_models.ResNet50_Weights.DEFAULT)
        backbone = nn.Sequential(
            m.conv1, m.bn1, m.relu, m.maxpool,
            m.layer1, m.layer2, m.layer3, m.layer4,
            m.avgpool,
            nn.Flatten(1),
        )
        feat_dim = 2048
        return backbone, feat_dim, 112

    raise ValueError(f'Unknown backbone: {name}. Choose mobilenetv2 / resnet18 / resnet50')


BACKBONE_NAMES = ['mobilenetv2', 'resnet18', 'resnet50']
print('Backbone factory ready. Available:', BACKBONE_NAMES)

Backbone factory ready. Available: ['mobilenetv2', 'resnet18', 'resnet50']


In [12]:
# ── Cell 12: CNN + LSTM Model ─────────────────────────────────────────────────
#
# Architecture
# ─────────────────────────────────────────────────────────────────────────────
#  Input : (B, T, C, H, W)   B=batch, T=seq_len, C/H/W = frame shape
#  Step 1: Reshape to (B·T, C, H, W) and pass through CNN backbone
#           → (B·T, feature_dim)
#  Step 2: Reshape back to (B, T, feature_dim)
#  Step 3: Feed sequence to a bidirectional LSTM
#           → hidden states (B, T, 2·hidden_size)
#  Step 4: Concatenate last forward & backward hidden states
#           → (B, 2·hidden_size)
#  Step 5: LayerNorm → Dropout → Linear → (B, num_classes)
# ─────────────────────────────────────────────────────────────────────────────

class CNNLSTMClassifier(nn.Module):
    """
    CNN feature extractor + bidirectional LSTM classifier for embryo
    phase classification from frame sequences.

    Parameters
    ----------
    backbone_name : str  — one of 'mobilenetv2', 'resnet18', 'resnet50'
    num_classes   : int
    hidden_size   : int  — LSTM hidden size (per direction)
    num_layers    : int  — stacked LSTM layers
    bidirectional : bool
    dropout       : float
    freeze_backbone : bool  — whether to freeze CNN weights during training
                              (fine-tuning = False gives better results but
                               uses more VRAM)
    """
    def __init__(
        self,
        backbone_name: str,
        num_classes: int  = NUM_CLASSES,
        hidden_size: int  = LSTM_HIDDEN,
        num_layers: int   = LSTM_LAYERS,
        bidirectional: bool = BIDIRECTIONAL,
        dropout: float    = DROPOUT,
        freeze_backbone: bool = False,
    ):
        super().__init__()
        self.backbone_name = backbone_name
        self.hidden_size   = hidden_size
        self.bidirectional = bidirectional

        # ── 1. CNN backbone ───────────────────────────────────────────────────
        self.backbone, feat_dim, self.img_size = get_backbone(backbone_name)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # Optional projection: reduce feature_dim before LSTM
        # (helps when backbone is large, e.g. ResNet50 = 2048)
        proj_dim = min(feat_dim, 512)
        self.projector = (
            nn.Sequential(
                nn.Linear(feat_dim, proj_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout * 0.5),
            )
            if feat_dim != proj_dim
            else nn.Identity()
        )
        lstm_input_dim = proj_dim

        # ── 2. Bidirectional LSTM ─────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size  = lstm_input_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            bidirectional = bidirectional,
            dropout = dropout if num_layers > 1 else 0.0,
        )

        # ── 3. Classification head ────────────────────────────────────────────
        lstm_out_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(lstm_out_dim),
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, T, C, H, W)
        returns logits : (B, num_classes)
        """
        B, T, C, H, W = x.shape

        # ── CNN: process all frames in parallel ───────────────────────────────
        x_flat   = x.view(B * T, C, H, W)           # (B·T, C, H, W)
        feats    = self.backbone(x_flat)              # (B·T, feat_dim)
        feats    = self.projector(feats)              # (B·T, proj_dim)
        feats    = feats.view(B, T, -1)              # (B, T, proj_dim)

        # ── LSTM ──────────────────────────────────────────────────────────────
        # lstm_out : (B, T, 2·hidden)  (bidirectional)
        # hn       : (num_layers·2, B, hidden)
        _, (hn, _) = self.lstm(feats)

        if self.bidirectional:
            # Last layer: forward hidden = hn[-2], backward hidden = hn[-1]
            context = torch.cat([hn[-2], hn[-1]], dim=1)  # (B, 2·hidden)
        else:
            context = hn[-1]                              # (B, hidden)

        # ── Head ──────────────────────────────────────────────────────────────
        return self.head(context)                         # (B, num_classes)


# ── Quick smoke test ──────────────────────────────────────────────────────────
print('Smoke test: MobileNetV2 + BiLSTM ...')
_m = CNNLSTMClassifier('mobilenetv2').to(DEVICE)
_x = torch.randn(2, SEQ_LEN, 3, 112, 112).to(DEVICE)  # 2 sequences
_y = _m(_x)
print(f'  Input  : {_x.shape}')
print(f'  Output : {_y.shape}   (expected (2, {NUM_CLASSES}))')
del _m, _x, _y
torch.cuda.empty_cache()

Smoke test: MobileNetV2 + BiLSTM ...
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 118MB/s] 


  Input  : torch.Size([2, 8, 3, 112, 112])
  Output : torch.Size([2, 15])   (expected (2, 15))


In [13]:
# ── Cell 13: Model Factory ────────────────────────────────────────────────────
# Convenience wrapper to build a CNNLSTMClassifier by backbone name and
# return the recommended image size alongside it.

def get_model(backbone_name: str,
              num_classes: int = NUM_CLASSES,
              freeze_backbone: bool = False):
    """
    Returns (model, img_size).

    freeze_backbone=True keeps CNN weights fixed (faster, lower VRAM).
    freeze_backbone=False fine-tunes the entire network end-to-end.
    """
    model    = CNNLSTMClassifier(backbone_name,
                                  num_classes     = num_classes,
                                  freeze_backbone = freeze_backbone)
    img_size = model.img_size
    return model, img_size


print('Model factory ready.')
print('Available backbones:', BACKBONE_NAMES)

Model factory ready.
Available backbones: ['mobilenetv2', 'resnet18', 'resnet50']


In [14]:
# ── Cell 14: Train & Evaluate Functions ──────────────────────────────────────

def train_epoch(model, loader, optimizer, loss_fn, clip=1.0):
    """
    One full training epoch.

    loader yields (sequences, labels) where sequences has shape
    (B, SEQ_LEN, C, H, W).
    """
    model.train()
    total_loss = 0.0

    for seqs, labels in tqdm(loader, desc='  train', leave=False):
        seqs   = seqs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(seqs)                     # (B, num_classes)
        loss   = loss_fn(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    """
    Returns (accuracy, weighted_f1) on the provided DataLoader.
    """
    model.eval()
    preds, targets = [], []

    for seqs, labels in tqdm(loader, desc='  eval ', leave=False):
        seqs = seqs.to(DEVICE, non_blocking=True)
        logits = model(seqs)                     # (B, num_classes)
        pred   = torch.argmax(logits, dim=1).cpu().numpy()
        preds.extend(pred)
        targets.extend(labels.numpy())

    acc = accuracy_score(targets, preds)
    f1  = f1_score(targets, preds, average='weighted', zero_division=0)
    return acc, f1


print('train_epoch() and evaluate() ready.')

train_epoch() and evaluate() ready.


In [15]:
# ── Cell 15: Training Driver ──────────────────────────────────────────────────

def run_experiment(backbone_name: str,
                   loss_fn,
                   label: str = '',
                   freeze_backbone: bool = False):
    """
    Train one CNN+LSTM model with a given loss function.

    Returns a result dict with val/test metrics and per-epoch history.
    """
    print(f'\n{"="*65}')
    print(f'  Backbone: {backbone_name.upper()}   Loss: {label}')
    print(f'  freeze_backbone={freeze_backbone}  '
          f'SEQ_LEN={SEQ_LEN}  LSTM_HIDDEN={LSTM_HIDDEN}')
    print(f'{"="*65}')

    model, img_size = get_model(backbone_name, freeze_backbone=freeze_backbone)
    model = model.to(DEVICE)

    # Count parameters
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters()
                           if p.requires_grad)
    print(f'  Total params: {total_params:,}  '
          f'Trainable: {trainable_params:,}')

    train_loader, val_loader, test_loader = make_loaders(img_size)

    optimizer = optim.AdamW(model.parameters(),
                             lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    history = []
    best_val_f1 = 0.0
    best_state  = None

    for epoch in range(NUM_EPOCHS):
        tr_loss         = train_epoch(model, train_loader, optimizer, loss_fn)
        val_acc, val_f1 = evaluate(model, val_loader)
        scheduler.step()

        history.append({
            'epoch'  : epoch + 1,
            'loss'   : tr_loss,
            'val_acc': val_acc,
            'val_f1' : val_f1,
        })
        print(f'  Ep {epoch+1}/{NUM_EPOCHS}  '
              f'loss={tr_loss:.4f}  '
              f'val_acc={val_acc:.4f}  '
              f'val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    # Restore best checkpoint
    model.load_state_dict(best_state)
    test_acc, test_f1 = evaluate(model, test_loader)
    print(f'  TEST  acc={test_acc:.4f}  f1={test_f1:.4f}')

    return {
        'model'       : backbone_name,
        'loss'        : label,
        'best_val_f1' : best_val_f1,
        'test_acc'    : test_acc,
        'test_f1'     : test_f1,
        'history'     : history,
        'state_dict'  : best_state,       # kept for saving
    }


print('run_experiment() ready.')

run_experiment() ready.


In [16]:
# ── Cell 16: Phase A — All backbones with SCCE baseline ───────────────────────

scce_loss = SCCELoss(CLASS_WEIGHTS)

results_scce = []
for name in BACKBONE_NAMES:
    res = run_experiment(name, scce_loss, label='SCCE')
    results_scce.append(res)

print('\n--- Phase A complete (SCCE) ---')


  Backbone: MOBILENETV2   Loss: SCCE
  freeze_backbone=False  SEQ_LEN=8  LSTM_HIDDEN=256
  Total params: 6,042,383  Trainable: 6,042,383


  Ep 1/5  loss=2.7044  val_acc=0.3398  val_f1=0.2814


  Ep 2/5  loss=2.2675  val_acc=0.2930  val_f1=0.2604


  Ep 3/5  loss=2.1086  val_acc=0.3789  val_f1=0.3951


  Ep 4/5  loss=1.8945  val_acc=0.4062  val_f1=0.4017


...

  Ep 2/5  loss=2.1933  val_acc=0.3320  val_f1=0.3035


  Ep 3/5  loss=1.9629  val_acc=0.3008  val_f1=0.3119


  Ep 4/5  loss=1.7886  val_acc=0.4453  val_f1=0.4459


  Ep 5/5  loss=1.6273  val_acc=0.3984  val_f1=0.4092


  TEST  acc=0.3264  f1=0.2855

--- Phase A complete (SCCE) ---


In [17]:
# ── Cell 17: Phase B — All backbones with Total Loss ──────────────────────────

total_loss = TotalLoss(CLASS_WEIGHTS, NUM_CLASSES, lam=LAMBDA)

results_total = []
for name in BACKBONE_NAMES:
    res = run_experiment(name, total_loss, label='SCCE+Ordinal')
    results_total.append(res)

print('\n--- Phase B complete (SCCE+Ordinal) ---')


  Backbone: MOBILENETV2   Loss: SCCE+Ordinal
  freeze_backbone=False  SEQ_LEN=8  LSTM_HIDDEN=256
  Total params: 6,042,383  Trainable: 6,042,383


  Ep 1/5  loss=2.8408  val_acc=0.2500  val_f1=0.2190


  Ep 2/5  loss=2.3687  val_acc=0.2930  val_f1=0.2513


  Ep 3/5  loss=2.1981  val_acc=0.3516  val_f1=0.3246


  Ep 4/5  loss=1.9740  val_acc=0.3516  val_f1=0.3593


...

  Ep 2/5  loss=2.3140  val_acc=0.3008  val_f1=0.2482


  Ep 3/5  loss=2.0827  val_acc=0.2070  val_f1=0.1912


  Ep 4/5  loss=1.8937  val_acc=0.4336  val_f1=0.4432


  Ep 5/5  loss=1.7132  val_acc=0.4453  val_f1=0.4530


  TEST  acc=0.3194  f1=0.2779

--- Phase B complete (SCCE+Ordinal) ---


In [18]:
# ── Cell 18: Comparison Table ─────────────────────────────────────────────────

rows = []
for r_a, r_b in zip(results_scce, results_total):
    delta_acc = r_b['test_acc'] - r_a['test_acc']
    delta_f1  = r_b['test_f1']  - r_a['test_f1']
    rows.append({
        'Backbone'       : r_a['model'].upper(),
        'Acc  (SCCE)'    : f"{r_a['test_acc']:.4f}",
        'Acc  (Total)'   : f"{r_b['test_acc']:.4f}",
        'ΔAcc'           : f"{delta_acc:+.4f}",
        'F1   (SCCE)'    : f"{r_a['test_f1']:.4f}",
        'F1   (Total)'   : f"{r_b['test_f1']:.4f}",
        'ΔF1'            : f"{delta_f1:+.4f}",
    })

comp_df = pd.DataFrame(rows)
print('\n====== Final Comparison — CNN+LSTM (Test Set) ======')
print(comp_df.to_string(index=False))

comp_df.to_csv('comparison_results_lstm.csv', index=False)
print('\nSaved → comparison_results_lstm.csv')


====== Final Comparison — CNN+LSTM (Test Set) ======
   Backbone Acc  (SCCE) Acc  (Total)    ΔAcc F1   (SCCE) F1   (Total)     ΔF1
MOBILENETV2      0.2778       0.2569 -0.0208      0.2414       0.2319 -0.0095
   RESNET18      0.3056       0.2743 -0.0312      0.2651       0.2394 -0.0257
   RESNET50      0.3264       0.3194 -0.0069      0.2855       0.2779 -0.0076

Saved → comparison_results_lstm.csv


In [19]:
# ── Cell 19: Per-Epoch Val F1 Summary ────────────────────────────────────────

print('\nPer-epoch Val F1 — SCCE vs Total Loss (CNN+LSTM)')
header = f"{'Epoch':>5}  "
for r in results_scce:
    header += f"{r['model'].upper()[:10]:>22}"
print(header)
print('-' * (5 + 22 * len(results_scce) + 10))

for ep in range(NUM_EPOCHS):
    line = f"  {ep+1:>3}    "
    for r_a, r_b in zip(results_scce, results_total):
        scce_f1  = r_a['history'][ep]['val_f1']
        total_f1 = r_b['history'][ep]['val_f1']
        line += f"  {scce_f1:.3f}/{total_f1:.3f}       "
    print(line)


Per-epoch Val F1 — SCCE vs Total Loss (CNN+LSTM)
Epoch              MOBILENETV              RESNET18              RESNET50
---------------------------------------------------------------------------------
    1      0.281/0.219         0.155/0.174         0.183/0.163       
    2      0.260/0.251         0.272/0.239         0.303/0.248       
    3      0.395/0.325         0.361/0.397         0.312/0.191       
    4      0.402/0.359         0.420/0.386         0.446/0.443       
    5      0.393/0.377         0.408/0.359         0.409/0.453       


In [20]:
# ── Cell 20: Classification Report — Best Model ───────────────────────────────
# Re-evaluate the best-performing (Phase B, Total Loss) model on the test set
# and print a full per-class classification report.

# Find best result by test_f1
best_result = max(results_total, key=lambda r: r['test_f1'])
print(f'Best model: {best_result["model"].upper()}  '
      f'(Loss: {best_result["loss"]})  '
      f'Test F1 = {best_result["test_f1"]:.4f}')

# Rebuild model and load best weights
best_model, img_size = get_model(best_result['model'])
best_model.load_state_dict(best_result['state_dict'])
best_model = best_model.to(DEVICE)
best_model.eval()

_, _, test_loader = make_loaders(img_size)

all_preds, all_targets = [], []
with torch.no_grad():
    for seqs, labels in tqdm(test_loader, desc='Final eval'):
        seqs   = seqs.to(DEVICE)
        logits = best_model(seqs)
        pred   = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(labels.numpy())

# Map label indices back to phase names for readability
idx_to_phase = {i: p for i, p in enumerate(PHASES[:NUM_CLASSES])}
target_names = [idx_to_phase.get(i, str(i)) for i in range(NUM_CLASSES)]

print('\n=== Classification Report ===')
print(classification_report(
    all_targets, all_preds,
    target_names=target_names,
    zero_division=0
))

Best model: RESNET50  (Loss: SCCE+Ordinal)  Test F1 = 0.2779


Final eval: 100%|██████████| 18/18 [00:04<00:00,  4.07it/s]


=== Classification Report ===
              precision    recall  f1-score   support

        tPB2       0.41      0.78      0.54         9
        tPNa       0.57      0.57      0.57        37
        tPNf       0.13      0.33      0.19         9
          t2       0.57      0.33      0.42        12
          t3       0.10      0.11      0.10        18
          t4       0.33      0.12      0.17        26
          t5       0.00      0.00      0.00        15
          t6       0.00      0.00      0.00        31
          t7       0.00      0.00      0.00        20
          t8       0.18      0.26      0.21        19
         t9+       0.37      0.59      0.46        39
          tM       0.31      0.33      0.32        12
         tSB       0.30      0.21      0.25        14
          tB       0.27      0.31      0.29        13
         tEB       0.43      0.93      0.59        14

    accuracy                           0.32       288
   macro avg       0.26      0.33      0.27      

In [21]:
# ── Cell 21: Save All Models ──────────────────────────────────────────────────

import os
os.makedirs('saved_models', exist_ok=True)

for phase_label, results in [('scce', results_scce),
                               ('total', results_total)]:
    for r in results:
        save_path = f"saved_models/{r['model']}_{phase_label}_lstm.pth"
        torch.save({
            'backbone_name' : r['model'],
            'loss'          : r['loss'],
            'num_classes'   : NUM_CLASSES,
            'seq_len'       : SEQ_LEN,
            'lstm_hidden'   : LSTM_HIDDEN,
            'lstm_layers'   : LSTM_LAYERS,
            'bidirectional' : BIDIRECTIONAL,
            'test_acc'      : r['test_acc'],
            'test_f1'       : r['test_f1'],
            'state_dict'    : r['state_dict'],
        }, save_path)
        print(f'Saved → {save_path}  '
              f'(test_f1={r["test_f1"]:.4f})')

print('\nAll models saved.')

Saved → saved_models/mobilenetv2_scce_lstm.pth  (test_f1=0.2414)
Saved → saved_models/resnet18_scce_lstm.pth  (test_f1=0.2651)
Saved → saved_models/resnet50_scce_lstm.pth  (test_f1=0.2855)
Saved → saved_models/mobilenetv2_total_lstm.pth  (test_f1=0.2319)
Saved → saved_models/resnet18_total_lstm.pth  (test_f1=0.2394)
Saved → saved_models/resnet50_total_lstm.pth  (test_f1=0.2779)

All models saved.


In [22]:
# ── Cell 22: Inference Helper (load saved model + predict) ───────────────────
# Shows how to reload a saved CNN+LSTM checkpoint and classify a new sequence
# of frames at inference time.

def load_model(ckpt_path: str) -> CNNLSTMClassifier:
    """Reload a CNN+LSTM model from a .pth checkpoint."""
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = CNNLSTMClassifier(
        backbone_name = ckpt['backbone_name'],
        num_classes   = ckpt['num_classes'],
        hidden_size   = ckpt['lstm_hidden'],
        num_layers    = ckpt['lstm_layers'],
        bidirectional = ckpt['bidirectional'],
    )
    model.load_state_dict(ckpt['state_dict'])
    model.to(DEVICE).eval()
    return model


def predict_sequence(model: CNNLSTMClassifier,
                     frame_paths: list,
                     img_size: int = 112) -> dict:
    """
    Classify a list of frame paths (length >= 1) as an embryo phase.

    If len(frame_paths) < SEQ_LEN the list is right-padded by repeating
    the last frame — identical to training-time padding.

    Returns a dict:
      predicted_label : int
      predicted_phase : str
      probabilities   : np.ndarray  (num_classes,)
    """
    eval_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Pad if necessary
    if len(frame_paths) < SEQ_LEN:
        frame_paths = frame_paths + [frame_paths[-1]] * (SEQ_LEN - len(frame_paths))
    else:
        frame_paths = frame_paths[:SEQ_LEN]

    frames = torch.stack([
        eval_tf(Image.open(p).convert('RGB')) for p in frame_paths
    ]).unsqueeze(0).to(DEVICE)   # (1, SEQ_LEN, C, H, W)

    with torch.no_grad():
        logits = model(frames)   # (1, num_classes)
        probs  = F.softmax(logits, dim=1).cpu().numpy()[0]
        pred   = int(probs.argmax())

    return {
        'predicted_label': pred,
        'predicted_phase': PHASES[pred],
        'probabilities'  : probs,
    }


print('load_model() and predict_sequence() helpers ready.')
print('Usage example (run after training):')
print('  model = load_model("saved_models/mobilenetv2_total_lstm.pth")')
print('  result = predict_sequence(model, ["frame1.jpg", "frame2.jpg", ...])')
print('  print(result["predicted_phase"])')

load_model() and predict_sequence() helpers ready.
Usage example (run after training):
  model = load_model("saved_models/mobilenetv2_total_lstm.pth")
  result = predict_sequence(model, ["frame1.jpg", "frame2.jpg", ...])
  print(result["predicted_phase"])


## Architecture Summary

```
Input sequence  (B, T=8, 3, 112, 112)
       │
       ▼  reshape → (B·T, 3, 112, 112)
┌─────────────────────────────────────┐
│  CNN Backbone (ImageNet pretrained) │  MobileNetV2 / ResNet18 / ResNet50
│  AdaptiveAvgPool2d(1) → Flatten     │
└──────────────── feat_dim ───────────┘
       │  (B·T, feat_dim)
       ▼  reshape → (B, T, feat_dim)
┌─────────────────────────────────────┐
│  Linear Projector (if feat_dim>512) │  feat_dim → 512
└─────────────────────────────────────┘
       │  (B, T, 512)
       ▼
┌─────────────────────────────────────┐
│  Bidirectional LSTM                 │  2 layers, hidden=256
│  → concat last fwd + bwd hidden     │  → (B, 512)
└─────────────────────────────────────┘
       │
       ▼
┌─────────────────────────────────────┐
│  LayerNorm → Dropout(0.3)           │
│  → Linear(512, 15)                  │
└─────────────────────────────────────┘
       │
  Logits  (B, 15 classes)
```

### Memory tip
Each batch holds `BATCH_SIZE × SEQ_LEN` images through the CNN simultaneously.  
With `BATCH_SIZE=16, SEQ_LEN=8` that is 128 forward passes per gradient step.  
If VRAM is tight: lower `BATCH_SIZE` to 8 **or** set `freeze_backbone=True` to cut the gradient graph.